# 03 — Pooled TCN / 1D-CNN Comparison

Compare TCN and 1D-CNN forecasts against the saved pooled LSTM and persistence baseline. Data preparation and split-local windowing are copied from `02_LSTM.ipynb`; no wind-direction features are included. Models train jointly on all stations, use train-only station scalers, and report test metrics in original concentration units.

## 1. Configuration

In [1]:

from pathlib import Path

LOOKBACK = 48
HORIZONS = [1, 6, 24]
BATCH_SIZE = 256
EPOCHS = 50
LR = 1e-3
HIDDEN = 64
LAYERS = 2
DROPOUT = 0.2
RANDOM_SEED = 42
EARLY_STOPPING_PATIENCE = 8
SCHEDULER_PATIENCE = 3

DATA_PATH = Path(r"C:\Users\jains\Downloads\Air quality dataset\cleaned\air_quality_all_stations_cleaned.csv")
RESULTS_DIR = Path.cwd() / "tcn_cnn_results"
LSTM_RESULTS_DIR = Path.cwd() / "lstm_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data: {DATA_PATH}")
print(f"Results: {RESULTS_DIR.resolve()}")
print(f"Saved LSTM results: {LSTM_RESULTS_DIR.resolve()}")


Data: C:\Users\jains\Downloads\Air quality dataset\cleaned\air_quality_all_stations_cleaned.csv
Results: E:\coding\DL2project\tcn_cnn_results
Saved LSTM results: E:\coding\DL2project\lstm_results


## 2. Imports, reproducibility, and device

In [2]:

import copy
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from IPython.display import display, Markdown
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def set_seed(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cpu":
    torch.set_num_threads(min(8, torch.get_num_threads()))
print(f"PyTorch: {torch.__version__}; device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


PyTorch: 2.15.0+cpu; device: cpu


## 3. Shared data and chronological splits (copied from 02_LSTM.ipynb)

In [3]:

if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Cleaned data file not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH, parse_dates=["datetime"])
FEATURES = [
    "PM10", "PM2.5", "temperature", "rain", "pressure", "precipitation",
    "wind_speed", "clouds_ordinal", "hour_sin", "hour_cos",
    "day_of_year_sin", "day_of_year_cos",
]
TARGETS = ["PM10", "PM2.5"]
required_columns = ["datetime", "station", *FEATURES]
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Required columns missing from cleaned data: {missing_columns}")
# Directional wind fields are intentionally excluded; wind_speed remains a feature.
df = df[required_columns].sort_values(["station", "datetime"]).reset_index(drop=True)
if df.duplicated(["station", "datetime"]).any():
    raise ValueError("Duplicate station/timestamp keys must be resolved before window creation.")

all_timestamps = pd.DatetimeIndex(df["datetime"].drop_duplicates().sort_values())
train_cut_idx = int(len(all_timestamps) * 0.70)
validation_cut_idx = int(len(all_timestamps) * 0.85)
if train_cut_idx <= 0 or validation_cut_idx <= train_cut_idx or validation_cut_idx >= len(all_timestamps):
    raise ValueError("Not enough distinct timestamps to create 70/15/15 chronological splits.")
TRAIN_START, TRAIN_END = all_timestamps[0], all_timestamps[train_cut_idx - 1]
VAL_START, VAL_END = all_timestamps[train_cut_idx], all_timestamps[validation_cut_idx - 1]
TEST_START, TEST_END = all_timestamps[validation_cut_idx], all_timestamps[-1]
print(f"Rows: {len(df):,}; stations: {df['station'].nunique()}; features: {len(FEATURES)}")
print(f"Train:      {TRAIN_START} through {TRAIN_END}")
print(f"Validation: {VAL_START} through {VAL_END}")
print(f"Test:       {TEST_START} through {TEST_END}")
print("Every station uses these same chronological cut dates.")
print(f"Wind-direction input columns retained: {[c for c in df.columns if 'wind_direction' in c]}")
expected_dates = [
    "2024-05-02 20:00:00", "2025-06-18 01:00:00",
    "2025-06-18 02:00:00", "2025-09-14 04:00:00",
    "2025-09-14 05:00:00", "2025-12-11 07:00:00",
]
actual_dates = [str(x) for x in [TRAIN_START, TRAIN_END, VAL_START, VAL_END, TEST_START, TEST_END]]
if actual_dates != expected_dates:
    raise ValueError(f"Split dates differ from requested/LSTM dates: {actual_dates}")


Rows: 296,100; stations: 21; features: 12
Train:      2024-05-02 20:00:00 through 2025-06-18 01:00:00
Validation: 2025-06-18 02:00:00 through 2025-09-14 04:00:00
Test:       2025-09-14 05:00:00 through 2025-12-11 07:00:00
Every station uses these same chronological cut dates.
Wind-direction input columns retained: []


## 4. Train-only scaling and split-local window datasets (copied from 02_LSTM.ipynb)

In [4]:

class WindowDataset(Dataset):
    '''Lazily slice windows so the full overlapping window tensor stays out of memory.'''
    def __init__(self, station_arrays, records, horizon):
        self.station_arrays = station_arrays
        self.records = records
        self.horizon = horizon

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        station_idx, origin_idx, target_idx = self.records[index]
        arrays = self.station_arrays[station_idx]
        x_start = origin_idx - LOOKBACK + 1
        x = arrays["x"][x_start:origin_idx + 1]
        y_scaled = arrays["y_scaled"][target_idx]
        y_original = arrays["y_original"][target_idx]
        persistence = arrays["persistence"][origin_idx]
        return (
            torch.from_numpy(x), torch.from_numpy(y_scaled),
            torch.tensor(station_idx, dtype=torch.long),
            torch.from_numpy(y_original), torch.from_numpy(persistence),
        )

def build_split_datasets(horizon):
    '''Fit station scalers on train only and create windows contained in each split.'''
    if horizon < 1:
        raise ValueError("HORIZON must be >= 1")
    split_bounds = {
        "train": (TRAIN_START, TRAIN_END),
        "validation": (VAL_START, VAL_END),
        "test": (TEST_START, TEST_END),
    }
    stations = sorted(df["station"].unique())
    station_arrays = {name: [] for name in split_bounds}
    records = {name: [] for name in split_bounds}
    station_scalers = {}
    for station_idx, station in enumerate(stations):
        station_df = df.loc[df["station"].eq(station)].sort_values("datetime").reset_index(drop=True)
        train_mask = station_df["datetime"].between(TRAIN_START, TRAIN_END)
        if train_mask.sum() <= LOOKBACK + horizon:
            raise ValueError(f"Station {station} has too few training rows for the configured window and horizon.")
        train_features = station_df.loc[train_mask, FEATURES].astype(float)
        medians = train_features.median()
        if medians.isna().any():
            bad = medians.index[medians.isna()].tolist()
            raise ValueError(f"Training features entirely missing for station {station}: {bad}")
        feature_scaler = StandardScaler()
        feature_scaler.fit(train_features.fillna(medians).to_numpy(dtype=np.float64))
        target_scalers = {}
        for target in TARGETS:
            valid_train_target = station_df.loc[train_mask, target].dropna().to_numpy(dtype=np.float64).reshape(-1, 1)
            if len(valid_train_target) == 0:
                raise ValueError(f"No valid training targets for {station} / {target}")
            target_scalers[target] = StandardScaler().fit(valid_train_target)
        station_scalers[station] = {
            "features": feature_scaler, "targets": target_scalers, "medians": medians
        }

        # Transform each split independently; no window can cross a split boundary.
        for split_name, (start, end) in split_bounds.items():
            part = station_df.loc[station_df["datetime"].between(start, end)].reset_index(drop=True)
            raw_x = part[FEATURES].astype(float).fillna(medians).to_numpy(dtype=np.float64)
            x_scaled = feature_scaler.transform(raw_x).astype(np.float32)
            y_original = part[TARGETS].to_numpy(dtype=np.float64)
            y_scaled = np.full(y_original.shape, np.nan, dtype=np.float32)
            persistence = part[TARGETS].to_numpy(dtype=np.float64)
            for target_idx, target in enumerate(TARGETS):
                valid = np.isfinite(y_original[:, target_idx])
                target_scaler = target_scalers[target]
                y_scaled[valid, target_idx] = target_scaler.transform(
                    y_original[valid, target_idx].reshape(-1, 1)
                ).ravel().astype(np.float32)
                # The last input observation is the persistence forecast origin.
                input_column_idx = FEATURES.index(target)
            # Preserve the source notebook's persistence arrays at the forecast origin.
            persistence[:, target_idx] = part[target].to_numpy(dtype=np.float64)
            local_arrays = {
                "x": x_scaled, "y_scaled": y_scaled,
                "y_original": y_original.astype(np.float32),
                "persistence": persistence.astype(np.float32),
            }
            station_arrays[split_name].append(local_arrays)
            first_target = LOOKBACK - 1 + horizon
            for target_pos in range(first_target, len(part)):
                origin_pos = target_pos - horizon
                if np.isfinite(y_original[target_pos]).all():
                    records[split_name].append((station_idx, origin_pos, target_pos))

    datasets = {
        name: WindowDataset(station_arrays[name], records[name], horizon)
        for name in split_bounds
    }
    return datasets, station_scalers, stations

print(f"Window builder ready: lookback={LOOKBACK}; horizons={HORIZONS}.")


Window builder ready: lookback=48; horizons=[1, 6, 24].


## 5. TCN, 1D-CNN, and LSTM reference architectures

In [5]:

class PooledLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=HIDDEN, layers=LAYERS, dropout=DROPOUT, output_size=2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size, hidden_size=hidden_size, num_layers=layers,
            batch_first=True, dropout=dropout if layers > 1 else 0.0,
        )
        self.head = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        sequence_output, _ = self.lstm(x)
        return self.head(sequence_output[:, -1, :])

class CausalResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation, dropout=DROPOUT, kernel_size=3):
        super().__init__()
        self.dilation = dilation
        self.kernel_size = kernel_size
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, dilation=dilation)
        self.norm1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, dilation=dilation)
        self.norm2 = nn.BatchNorm1d(out_channels)
        self.dropout = nn.Dropout(dropout)
        self.residual = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def _causal_conv(self, x, conv):
        return conv(F.pad(x, (self.dilation * (self.kernel_size - 1), 0)))

    def forward(self, x):
        residual = self.residual(x)
        out = self.dropout(F.relu(self.norm1(self._causal_conv(x, self.conv1))))
        out = self.dropout(self.norm2(self._causal_conv(out, self.conv2)))
        return F.relu(out + residual)

class PooledTCN(nn.Module):
    def __init__(self, input_size, channels=64, output_size=2):
        super().__init__()
        # Dilations 1, 2, 4, 8, 16 with two kernel-3 convolutions cover >48 steps.
        dilations = [1, 2, 4, 8, 16]
        blocks = []
        in_channels = input_size
        for dilation in dilations:
            blocks.append(CausalResidualBlock(in_channels, channels, dilation))
            in_channels = channels
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Linear(channels, output_size)

    def forward(self, x):
        sequence = self.blocks(x.transpose(1, 2))
        return self.head(sequence[:, :, -1])

class PooledCNN1D(nn.Module):
    def __init__(self, input_size, output_size=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(input_size, 32, 3, padding=1),
            nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Conv1d(32, 64, 3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Conv1d(64, 64, 3, padding=1),
            nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(DROPOUT),
        )
        self.head = nn.Linear(64, output_size)

    def forward(self, x):
        return self.head(self.features(x.transpose(1, 2)).mean(dim=2))

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

MODEL_FACTORIES = {
    "TCN": lambda: PooledTCN(len(FEATURES)),
    "1D-CNN": lambda: PooledCNN1D(len(FEATURES)),
    "LSTM": lambda: PooledLSTM(len(FEATURES)),
}
PARAM_COUNTS = {name: count_parameters(factory()) for name, factory in MODEL_FACTORIES.items()}
print(pd.DataFrame([
    {"model": name, "trainable_parameters": count}
    for name, count in PARAM_COUNTS.items()
]).to_string(index=False))


 model  trainable_parameters
   TCN                115778
1D-CNN                 20194
  LSTM                 53378


## 6. Training, early stopping, original-scale evaluation, and saved outputs

In [ ]:

def regression_metrics(actual, predicted):
    actual, predicted = np.asarray(actual, dtype=float), np.asarray(predicted, dtype=float)
    valid = np.isfinite(actual) & np.isfinite(predicted)
    actual, predicted = actual[valid], predicted[valid]
    if not len(actual):
        return {"MAE": np.nan, "RMSE": np.nan, "R2": np.nan, "n": 0}
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": r2_score(actual, predicted) if len(actual) >= 2 else np.nan,
        "n": len(actual),
    }

lstm_reference_path = LSTM_RESULTS_DIR / "lstm_all_horizons_metrics.csv"
if not lstm_reference_path.is_file():
    raise FileNotFoundError(f"Saved pooled LSTM metrics not found: {lstm_reference_path}")
lstm_reference = pd.read_csv(lstm_reference_path)
required_cols = {"horizon", "target", "station", "method", "MAE", "RMSE", "R2", "n"}
if not required_cols.issubset(lstm_reference.columns):
    raise ValueError(f"Unexpected LSTM metrics columns: {lstm_reference.columns.tolist()}")

training_records = []
model_metrics = {"TCN": [], "1D-CNN": []}
model_predictions = {"TCN": {}, "1D-CNN": {}}
for model_name in ("TCN", "1D-CNN"):
    model_started = time.time()
    model_key = "tcn" if model_name == "TCN" else "1d_cnn"
    for horizon in HORIZONS:
        set_seed(RANDOM_SEED)
        datasets, station_scalers, stations = build_split_datasets(horizon)
        if any(len(datasets[name]) == 0 for name in ("train", "validation", "test")):
            raise ValueError(f"At least one split has no valid windows for horizon {horizon}.")
        window_counts = {name: len(ds) for name, ds in datasets.items()}
        reference_test_counts = lstm_reference.loc[
            lstm_reference["horizon"].eq(horizon)
            & lstm_reference["station"].eq("overall")
            & lstm_reference["method"].eq("LSTM"), "n"
        ]
        if len(reference_test_counts) != 2 or not (reference_test_counts.astype(int) == window_counts["test"]).all():
            raise ValueError(
                f"Test windows {window_counts['test']} do not match pooled LSTM counts "
                f"for H={horizon}: {reference_test_counts.tolist()}"
            )
        print(
            f"{model_name} H={horizon}: windows train/validation/test = "
            f"{window_counts['train']:,}/{window_counts['validation']:,}/{window_counts['test']:,}; "
            "test count matches saved LSTM pooled metrics."
        )
        train_loader = DataLoader(
            datasets["train"], batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
            pin_memory=DEVICE.type == "cuda",
            generator=torch.Generator().manual_seed(RANDOM_SEED),
        )
        val_loader = DataLoader(
            datasets["validation"], batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
            pin_memory=DEVICE.type == "cuda",
        )
        test_loader = DataLoader(
            datasets["test"], batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
            pin_memory=DEVICE.type == "cuda",
        )
        model = (PooledTCN(len(FEATURES)) if model_name == "TCN" else PooledCNN1D(len(FEATURES))).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
        criterion = nn.MSELoss()
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=SCHEDULER_PATIENCE
        )
        best_val_loss, best_state, best_epoch = float("inf"), None, 0
        no_improvement = 0
        history = {"train_loss": [], "validation_loss": []}
        weights_path = RESULTS_DIR / f"{model_key}_horizon_{horizon}_best.pt"
        for epoch in range(1, EPOCHS + 1):
            model.train()
            train_sum, train_count = 0.0, 0
            for x, y, _, _, _ in train_loader:
                x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                loss = criterion(model(x), y)
                loss.backward()
                optimizer.step()
                train_sum += loss.item() * len(x)
                train_count += len(x)
            train_loss = train_sum / train_count
            model.eval()
            val_sum, val_count = 0.0, 0
            with torch.no_grad():
                for x, y, _, _, _ in val_loader:
                    x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
                    val_sum += criterion(model(x), y).item() * len(x)
                    val_count += len(x)
            val_loss = val_sum / val_count
            scheduler.step(val_loss)
            history["train_loss"].append(train_loss)
            history["validation_loss"].append(val_loss)
            if val_loss < best_val_loss:
                best_val_loss, best_epoch = val_loss, epoch
                best_state = copy.deepcopy(model.state_dict())
                torch.save(best_state, weights_path)
                no_improvement = 0
            else:
                no_improvement += 1
            print(
                f"{model_name} H={horizon:>2} | epoch {epoch:02d}/{EPOCHS} | "
                f"train={train_loss:.5f} | val={val_loss:.5f} | "
                f"lr={optimizer.param_groups[0]['lr']:.2e}"
            )
            if no_improvement >= EARLY_STOPPING_PATIENCE:
                print(f"Early stopping after {EARLY_STOPPING_PATIENCE} epochs without improvement.")
                break
        if best_state is None:
            raise RuntimeError(f"No finite validation checkpoint: {model_name} H={horizon}")
        model.load_state_dict(torch.load(weights_path, map_location=DEVICE, weights_only=True))
        model.eval()
        pred_scaled, actual, persistence, station_ids = [], [], [], []
        with torch.no_grad():
            for x, _, sid, y_original, persistence_original in test_loader:
                pred_scaled.append(model(x.to(DEVICE, non_blocking=True)).cpu().numpy())
                actual.append(y_original.numpy())
                persistence.append(persistence_original.numpy())
                station_ids.append(sid.numpy())
        pred_scaled = np.concatenate(pred_scaled)
        actual = np.concatenate(actual)
        persistence = np.concatenate(persistence)
        station_ids = np.concatenate(station_ids)
        station_names = np.asarray(stations, dtype=object)[station_ids]
        pred_original = np.empty_like(pred_scaled, dtype=np.float64)
        for station_idx, station in enumerate(stations):
            rows = station_ids == station_idx
            for target_idx, target in enumerate(TARGETS):
                pred_original[rows, target_idx] = station_scalers[station]["targets"][target].inverse_transform(
                    pred_scaled[rows, target_idx].reshape(-1, 1)
                ).ravel()

        test_dates = []
        # Dataset order is station-major, then chronological within each station.
        for station in stations:
            part = df.loc[
                df["station"].eq(station) & df["datetime"].between(TEST_START, TEST_END)
            ].reset_index(drop=True)
            first_target = LOOKBACK - 1 + horizon
            valid_positions = [
                pos for pos in range(first_target, len(part))
                if np.isfinite(part.loc[pos, TARGETS].to_numpy(dtype=float)).all()
            ]
            test_dates.extend(part.loc[valid_positions, "datetime"].tolist())
        prediction_frame = pd.DataFrame({
            "station": station_names, "datetime": test_dates,
            "PM10_actual": actual[:, 0], "PM10_predicted": pred_original[:, 0],
            "PM2.5_actual": actual[:, 1], "PM2.5_predicted": pred_original[:, 1],
        })
        model_predictions[model_name][horizon] = prediction_frame
        prediction_frame.to_csv(
            RESULTS_DIR / f"{model_key}_horizon_{horizon}_test_predictions.csv", index=False
        )
        for target_idx, target in enumerate(TARGETS):
            for group, mask in [("overall", np.ones(len(station_names), dtype=bool))] + [
                (station, station_names == station) for station in stations
            ]:
                values = regression_metrics(actual[mask, target_idx], pred_original[mask, target_idx])
                model_metrics[model_name].append({
                    "horizon": horizon, "target": target, "station": group,
                    "method": model_name, **values,
                })
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(history["train_loss"], label="Train")
        ax.plot(history["validation_loss"], label="Validation")
        ax.set(
            title=f"{model_name} training history — horizon {horizon}",
            xlabel="Epoch", ylabel="MSE (scaled targets)",
        )
        ax.legend()
        fig.tight_layout()
        fig.savefig(RESULTS_DIR / f"{model_key}_horizon_{horizon}_loss.png", dpi=150)
        plt.show()
        training_records.append({
            "model": model_name, "horizon": horizon, "best_epoch": best_epoch,
            "best_train_loss": history["train_loss"][best_epoch - 1],
            "best_validation_loss": best_val_loss, "epochs_run": len(history["train_loss"]),
            "training_seconds_model_total": np.nan,
        })
        print(f"{model_name} H={horizon}: best epoch {best_epoch}; best val MSE={best_val_loss:.6f}")
    elapsed = time.time() - model_started
    for record in training_records:
        if record["model"] == model_name:
            record["training_seconds_model_total"] = elapsed

for name, rows in model_metrics.items():
    key = "tcn" if name == "TCN" else "1d_cnn"
    metrics_frame = pd.DataFrame(rows)
    metrics_frame.to_csv(RESULTS_DIR / f"{key}_all_horizons_metrics.csv", index=False)
    print(f"Saved pooled and per-station {name} metrics: {len(metrics_frame):,} rows")
training_summary = pd.DataFrame(training_records)
display(training_summary)


## 7. Pooled comparison, per-station analysis, and plots

In [ ]:

# LSTM file contains both the pooled LSTM and persistence rows.
lstm_pooled = lstm_reference.loc[lstm_reference["station"].eq("overall")].copy()
learned = pd.concat(
    [pd.DataFrame(model_metrics["TCN"]), pd.DataFrame(model_metrics["1D-CNN"])],
    ignore_index=True,
)
learned = learned.loc[learned["station"].eq("overall")]
comparison_long = pd.concat([lstm_pooled, learned], ignore_index=True)
method_order = ["Persistence", "LSTM", "TCN", "1D-CNN"]
metric_order = ["MAE", "RMSE", "R2"]
comparison_long = comparison_long.loc[comparison_long["method"].isin(method_order)]
wide = comparison_long.pivot_table(
    index=["target", "horizon"], columns="method",
    values=metric_order, aggfunc="first",
)
wide = wide.reindex(columns=pd.MultiIndex.from_product([metric_order, method_order]))
wide.columns = [f"{method}_{metric}" for metric, method in wide.columns]
comparison_wide = wide.reset_index().sort_values(["target", "horizon"]).reset_index(drop=True)
comparison_wide.to_csv(RESULTS_DIR / "model_comparison_results.csv", index=False)
print(f"Saved comparison table: {(RESULTS_DIR / 'model_comparison_results.csv').resolve()}")

def highlight_best(row):
    styles = [""] * len(row)
    positions = {name: idx for idx, name in enumerate(row.index)}
    for metric in metric_order:
        columns = [f"{method}_{metric}" for method in method_order]
        values = pd.to_numeric(row[columns], errors="coerce")
        best = values.max() if metric == "R2" else values.min()
        for column in columns:
            value = pd.to_numeric(pd.Series([row[column]]), errors="coerce").iloc[0]
            if pd.notna(value) and np.isclose(value, best):
                styles[positions[column]] = "background-color: #b7e4c7; font-weight: bold"
    return styles

display(comparison_wide.style.apply(highlight_best, axis=1).format(precision=3))

for metric in ("RMSE", "R2"):
    fig, axes = plt.subplots(2, 3, figsize=(17, 8), squeeze=False)
    for row_idx, target in enumerate(TARGETS):
        for col_idx, horizon in enumerate(HORIZONS):
            ax = axes[row_idx, col_idx]
            selected = comparison_wide.loc[
                comparison_wide["target"].eq(target) & comparison_wide["horizon"].eq(horizon)
            ]
            if selected.empty:
                raise ValueError(f"Missing comparison row for {target}, H={horizon}")
            vals = [selected[f"{method}_{metric}"].iloc[0] for method in method_order]
            ax.bar(method_order, vals, color=["#8c8c8c", "#4c78a8", "#f58518", "#54a24b"])
            ax.set_title(f"{target} | H={horizon}")
            ax.set_ylabel(metric)
            ax.tick_params(axis="x", rotation=20)
    fig.suptitle(f"Test {metric} by target, horizon, and method")
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / f"comparison_{metric.lower()}_bars.png", dpi=160)
    plt.show()

per_station = comparison_long.loc[
    comparison_long["horizon"].eq(6) & comparison_long["target"].eq("PM2.5")
    & comparison_long["station"].ne("overall")
    & comparison_long["method"].isin(["LSTM", "TCN", "1D-CNN"]),
    ["station", "method", "RMSE"],
]
station_rmse = per_station.pivot(index="station", columns="method", values="RMSE")
station_rmse = station_rmse.reindex(columns=["LSTM", "TCN", "1D-CNN"])
if station_rmse.isna().any().any():
    raise ValueError("Per-station PM2.5 RMSE comparison is incomplete at H=6")
fig, ax = plt.subplots(figsize=(13, 6))
image = ax.imshow(station_rmse.to_numpy(), aspect="auto", cmap="magma")
ax.set_xticks(range(len(station_rmse.columns)), labels=station_rmse.columns)
ax.set_yticks(range(len(station_rmse.index)), labels=station_rmse.index)
ax.set_title("Per-station PM2.5 test RMSE (H=6)")
fig.colorbar(image, ax=ax, label="RMSE")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "pm25_h6_per_station_rmse_heatmap.png", dpi=160)
plt.show()

# One-week H=6 test slice for PM2.5 at the first two stations, for all models.
lstm_path = LSTM_RESULTS_DIR / "lstm_horizon_6_test_predictions.csv"
if not lstm_path.is_file():
    raise FileNotFoundError(f"Saved LSTM test predictions not found: {lstm_path}")
lstm_predictions = pd.read_csv(lstm_path, parse_dates=["datetime"])
plot_stations = stations[:2]
plot_start, plot_end = TEST_START, min(TEST_END, TEST_START + pd.Timedelta(days=7))
fig, axes = plt.subplots(3, len(plot_stations), figsize=(16, 11), squeeze=False)
frames = [
    ("LSTM", lstm_predictions),
    ("TCN", model_predictions["TCN"][6]),
    ("1D-CNN", model_predictions["1D-CNN"][6]),
]
for row_idx, (model_name, frame) in enumerate(frames):
    frame = frame.copy()
    frame["datetime"] = pd.to_datetime(frame["datetime"])
    for col_idx, station in enumerate(plot_stations):
        plotted = frame.loc[
            frame["station"].eq(station) & frame["datetime"].between(plot_start, plot_end)
        ].sort_values("datetime")
        if plotted.empty:
            raise ValueError(f"No H=6 predictions for station {station}, model {model_name}")
        ax = axes[row_idx, col_idx]
        ax.plot(plotted["datetime"], plotted["PM2.5_actual"], label="Actual", linewidth=1.5)
        ax.plot(plotted["datetime"], plotted["PM2.5_predicted"], label=model_name, linewidth=1.2)
        ax.set_title(f"{model_name} — {station} — PM2.5, H=6")
        ax.tick_params(axis="x", rotation=30)
        if row_idx == 0 and col_idx == 0:
            ax.legend()
fig.suptitle("One-week test slice: actual vs predicted PM2.5")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "h6_one_week_actual_vs_predicted.png", dpi=160)
plt.show()

time_rows = []
for name in ("TCN", "1D-CNN"):
    seconds = float(training_summary.loc[
        training_summary["model"].eq(name), "training_seconds_model_total"
    ].iloc[0])
    time_rows.append({
        "model": name, "trainable_parameters": PARAM_COUNTS[name],
        "training_time_seconds": seconds,
    })
time_rows.append({
    "model": "LSTM (reference)", "trainable_parameters": PARAM_COUNTS["LSTM"],
    "training_time_seconds": np.nan,
})
time_table = pd.DataFrame(time_rows)
display(time_table)
time_table.to_csv(RESULTS_DIR / "training_time_parameter_counts.csv", index=False)

lines = ["### Summary", ""]
for target in TARGETS:
    for horizon in HORIZONS:
        row = comparison_wide.loc[
            comparison_wide["target"].eq(target) & comparison_wide["horizon"].eq(horizon)
        ].iloc[0]
        rmse = {method: row[f"{method}_RMSE"] for method in method_order}
        winner = min(rmse, key=rmse.get)
        improvement = 100 * (row["Persistence_RMSE"] - rmse[winner]) / row["Persistence_RMSE"]
        lines.append(
            f"- **{target}, H={horizon}:** {winner} has the lowest RMSE ({rmse[winner]:.3f}), "
            f"{improvement:.1f}% lower than persistence."
        )
lines.extend(["", "Overfitting signals (scaled-target loss):"])
for _, record in training_summary.iterrows():
    gap = record["best_validation_loss"] - record["best_train_loss"]
    ratio = record["best_validation_loss"] / max(record["best_train_loss"], 1e-12)
    flags = []
    if record["best_epoch"] <= 5:
        flags.append("early best epoch")
    if ratio >= 1.5:
        flags.append("validation loss at least 1.5x train loss")
    signal = "; ".join(flags) if flags else "no strong early-stop / train-validation-gap signal"
    lines.append(
        f"- {record['model']} H={int(record['horizon'])}: best epoch "
        f"{int(record['best_epoch'])}/{int(record['epochs_run'])}; loss gap {gap:.4f} "
        f"(ratio {ratio:.2f}) — {signal}."
    )
display(Markdown("\\n".join(lines)))
print(f"Outputs saved under {RESULTS_DIR.resolve()}")
